# Montandon: Floods

This notebook fetches recent flood data from Montandon, the global crisis data bank, filtering for severity, and visualizes them on a map.

In [1]:
import os
import pandas as pd
from pystac_client import Client
import geopandas as gpd
from shapely.geometry import Point, Polygon, shape
from lonboard import viz

from datetime import datetime, timedelta, timezone

## Connect to Montandon STAC

Montandon is exposed as a STAC collection, but requires authentication.

In [2]:
STAC_API_URL = "https://montandon-eoapi.ifrc.org/stac"
API_TOKEN = os.getenv('MONTANDON_API_TOKEN')

In [3]:
auth_headers = {"Authorization": f"Bearer {API_TOKEN}"}

### Check Auth

In [4]:
# Connect to STAC API with authentication
try:
    client = Client.open(STAC_API_URL, headers=auth_headers)
    print(f"\n[OK] Connected to: {STAC_API_URL}")
    print(f"[OK] API Title: {client.title}")
    print(f"[OK] Authentication: Bearer Token (OpenID Connect)")
except Exception as e:
    print(f"\n[ERROR] Authentication failed: {e}")


[OK] Connected to: https://montandon-eoapi.ifrc.org/stac
[OK] API Title: Montandon STAC API
[OK] Authentication: Bearer Token (OpenID Connect)


In [5]:
client

<Client id=montandon-eoapi>

## Fetch most recent Events

List collections with event items, usig simple string-matching on the collection id

In [6]:
collections_list = list(client.get_collections())

In [7]:
events_collection = [c for c in collections_list if '-events' in c.id]

In [8]:
for col in events_collection:
    print(f"{col.title} | {col.id}")

DesInventar Mapped Events | desinventar-events
EM-DAT Source Events | emdat-events
GDACS Source Events | gdacs-events
GFD Source Events | gfd-events
GLIDE Source Events | glide-events
IBTrACS Source Events | ibtracs-events
IDMC GIDD Source Events | idmc-gidd-events
IDMC Internal Displacement Updates (IDU) Impacts | idmc-idu-events
IFRC Source Events | ifrcevent-events
PDC Source Events | pdc-events
USGS Events | usgs-events


### Floods: GDACS Hazards
Search `gdacs-hazards` for recent flood events, filtering by hazard code `FL` in Python.

In [9]:
now = datetime.now(timezone.utc)
one_week_ago = now - timedelta(hours=168)

In [19]:
search_hazards = client.search(
    collections=["gdacs-hazards"],
    datetime=f"{one_week_ago.isoformat()}/{now.isoformat()}",
    max_items=1000,
)

flood_hazards = [
    item for item in search_hazards.items()
    if 'MH0600' in item.properties.get('monty:hazard_codes', [])
]
print(f"Found {len(flood_hazards)} flood hazard records")

Found 61 flood hazard records


In [20]:
len(flood_hazards)

61

In [21]:
floods = gpd.GeoDataFrame(
    [{
        'id': item.id,
        'monty_corr_id': item.properties['monty:corr_id'],
        'start_datetime': pd.to_datetime(item.properties.get('start_datetime')),
        'title': item.properties.get('title'),
        'country_codes': item.properties.get('monty:country_codes'),
        'severity_value': item.properties.get('monty:hazard_detail', {}).get('severity_value'),
        'severity_label': item.properties.get('monty:hazard_detail', {}).get('severity_label'),
        'geometry': shape(item.geometry),
    } for item in flood_hazards],
    geometry='geometry',
    crs='EPSG:4326',
).sort_values('start_datetime', ascending=False)

In [22]:
floods

,id,monty_corr_id,start_datetime,title,country_codes,severity_value,severity_label,geometry
0,gdacs-hazard-1103888-1,20260519-USA-1161474-MH0600-1-GCDB,2026-05-19 01:00:00+00:00,Flood in United States,[USA],0.5,Green,"POLYGON ((-85.4007 39.0812, -85.3681 39.0678, ..."
1,gdacs-hazard-1103887-1,20260518-AUS-556064-MH0600-1-GCDB,2026-05-18 01:00:00+00:00,Flood in Australia,[AUS],0.5,Green,"POLYGON ((152.7325 -28.2175, 152.7022 -28.3378..."
2,gdacs-hazard-1103884-1,20260516-MYS-824018-MH0600-1-GCDB,2026-05-16 01:00:00+00:00,Flood in Malaysia,[MYS],0.5,Green,"POLYGON ((103.3902 1.6221, 103.4803 1.6221, 10..."
3,gdacs-hazard-1103883-4,20260515-CHN-1077856-MH0600-4-GCDB,2026-05-15 01:00:00+00:00,Flood in China,[CHN],0.5,Green,"POLYGON ((111.3893 29.2735, 111.5378 29.5887, ..."
4,gdacs-hazard-1103883-3,20260515-CHN-1077856-MH0600-3-GCDB,2026-05-15 01:00:00+00:00,Flood in China,[CHN],0.5,Green,"POLYGON ((108.1457 24.978, 108.2358 24.9758, 1..."
...,...,...,...,...,...,...,...,...
29,gdacs-hazard-1103786-38,20260302-IDN-816913-MH0600-38-GCDB,2026-03-02 01:00:00+00:00,Flood in Indonesia,[IDN],0.5,Green,"POLYGON ((107.5057 -7.0077, 107.5957 -7.0122, ..."
31,gdacs-hazard-1103786-36,20260302-IDN-816913-MH0600-36-GCDB,2026-03-02 01:00:00+00:00,Flood in Indonesia,[IDN],0.5,Green,"POLYGON ((102.4053 -3.5946, 102.4954 -3.5968, ..."
32,gdacs-hazard-1103786-35,20260302-IDN-816913-MH0600-35-GCDB,2026-03-02 01:00:00+00:00,Flood in Indonesia,[IDN],0.5,Green,"POLYGON ((104.123 -4.1261, 104.213 -4.1217, 10..."
33,gdacs-hazard-1103786-34,20260302-IDN-816913-MH0600-34-GCDB,2026-03-02 01:00:00+00:00,Flood in Indonesia,[IDN],0.5,Green,"POLYGON ((120.9529 0.4417, 120.9535 0.4415, 12..."


In [23]:
print(f"Found {len(floods)} recent flood hazards with {len(floods['monty_corr_id'].unique())} unique Montandon correlation IDs.")

Found 61 recent flood hazards with 61 unique Montandon correlation IDs.


In [25]:
viz(floods)